In [5]:
import os
import requests
import pandas as pd
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())

In [6]:

#EXTRAE HASTA 100 RESULTADOS

query = "f1"
country = "ES"
leng = "ES"

def google_custom_search_df(query, country="ES", max_requests=10):
    API_KEY = os.getenv('API_KEY')
    API_CUSTOM_SEARCH_ID = os.getenv('API_CUSTOM_SEARCH_ID')

    if not API_KEY or not API_CUSTOM_SEARCH_ID:
        print("Error: API Key o ID de búsqueda no están cargados correctamente")
        return pd.DataFrame()

    search_results = []
    for i in range(1, max_requests * 10, 10):  # Limita a 10 resultados por búsqueda, hasta max_requests * 10
        url = f"https://www.googleapis.com/customsearch/v1?key={API_KEY}&cx={API_CUSTOM_SEARCH_ID}&q={query}&start={i}&gl={country}"
        response = requests.get(url)
        
        if response.status_code != 200:
            print(f"Error en la petición: {response.status_code}")
            break  # Detenemos las solicitudes si hay un error
        
        data = response.json()
        items = data.get("items", [])
        
        if not items:
            print("No se encontraron más resultados")
            break  # Detenemos si no hay más resultados
        
        search_results.extend(items)

    # Crear un DataFrame con los resultados de la búsqueda
    if search_results:
        df = pd.DataFrame(search_results, columns=['title', 'link'])
    else:
        df = pd.DataFrame()

    return df

# Llamada a la función con un límite de 10 requests (equivalente a 100 resultados)
df = google_custom_search_df(query, country, max_requests=10)
print(df)

KeyboardInterrupt: 

In [7]:
#EXTRAE SOLO 10 RESULTADOS

query = "autopublicar"
country = "ES"

def google_custom_search_df(query, country):
    API_KEY = os.getenv('API_KEY')
    API_CUSTOM_SEARCH_ID = os.getenv('API_CUSTOM_SEARCH_ID')

    if not API_KEY or not API_CUSTOM_SEARCH_ID:
        print("Error: API Key o ID de búsqueda no están cargados correctamente")
        return pd.DataFrame()

    # URL request para obtener solo los primeros 10 resultados
    url = f"https://www.googleapis.com/customsearch/v1?key={API_KEY}&cx={API_CUSTOM_SEARCH_ID}&q={query}&start=1&gl={country}"
    response = requests.get(url)
    
    if response.status_code != 200:
        print(f"Error en la petición: {response.status_code}")
        return pd.DataFrame()

    data = response.json()

    # Extraer los primeros 10 resultados
    items = data.get("items", [])

    if not items:
        print("No se encontraron resultados")
        return pd.DataFrame()

    # Crear un DataFrame con los resultados de la búsqueda
    df = pd.DataFrame(items, columns=['title', 'link'])

    return df

# Llamada a la función para obtener solo los primeros 10 resultados
df = google_custom_search_df(query, country)
print(df)

                                               title  \
0  Autopublicación | Amazon Kindle Direct Publishing   
1  Cómo autopublicar un libro | Gratis y online |...   
2  Los 10 errores más comunes al autopublicar un ...   
3    Autopublicar un libro ya publicado en editorial   
4                 Autopublicar libros en Puerto Rico   
5  ¿Cómo autopublicar un libro de forma independi...   
6    ¿Deberías autopublicar tu libro? | Entrepreneur   
7  Autopublicar desde principiante a Profesional ...   
8  Cómo autopublicar un libro en 8 pasos (2024) -...   
9  FundéuRAE on X: "@2404km Sí, «autopublicar» es...   

                                                link  
0                      https://kdp.amazon.com/es_ES/  
1  https://www.bookmundo.com/es/publica-tu-libro/...  
2  https://marketingonlineparaescritores.com/erro...  
3  https://www.kdpcommunity.com/s/question/0D58V0...  
4  https://www.facebook.com/events/d41d8cd9/autop...  
5  https://www.escritores.org/publicar/1275-como-... 

In [8]:
#-------------------SCRAPING del texto de los articulos de las 10 primeras posiciones de Google SERP

#Extraer el texto de los articulos con la libreria newspapper y, en su defecto, con beautiful soup
from bs4 import BeautifulSoup
from newspaper import Article

def scrape_article(url):
    try:
        article = Article(url)
        article.download()
        article.parse()
        if not article.text:
            # Si Newspaper no pudo obtener el texto, intenta con BeautifulSoup
            page = requests.get(url)
            soup = BeautifulSoup(page.content, 'html.parser')
            
            # Busca contenido en etiquetas <p> o <div>
            article_text = ' '.join([p.get_text() for p in soup.find_all(['p', 'div'])])
            
            return article_text
        return article.text
    except Exception as e:
        print(f"Error al scrapear el artículo: {str(e)}")
        return ""

def scrape_articles_in_dataframe(df):
    scraped_texts = []  # Aquí almacenaremos el texto de los artículos

    for url in df['link']:
        scraped_text = scrape_article(url)
        scraped_texts.append(scraped_text)

    # Agregamos los textos como una nueva columna en el DataFrame
    df['scraped_text'] = scraped_texts

    return df

# Llama a la función scraping de los artículos
df = scrape_articles_in_dataframe(df)

In [9]:
from openai import OpenAI

client = OpenAI()
import time

# Función para hacer la llamada a GPT-4 y obtener un resumen y análisis SEO
def gpt4_summarize_and_analyze(text):
    try:
        response = client.chat.completions.create(model="gpt-4",
        messages=[
        {"role": "system", "content": "You are an expert in semantic SEO optimization and content strategy."},
        {"role": "user", "content": f"Summarize the following article and extract the most important points related to SEO, including the keywords, biagrams, triagrams, structure, meta tags, and content strategy used. Focus on identifying strengths and weaknesses in SEO practices to determine what makes this article rank highly in search engines: {text}. Keep in mind that all this analysis will serve as a basis for the creation of new content that will outperform them in SEO. Output: Spanish."}
])
        return response.choices[0].message.content
    except Exception as e:
        print(f"Error al generar el análisis con GPT-4: {str(e)}")
        return "Error al generar análisis con GPT-4"

# Función principal para procesar los artículos scrapeados
def process_scraped_articles(df):
    if df.empty:
        print("El DataFrame está vacío, no hay datos para procesar.")
        return df

    for index, row in df.iterrows():
        article_text = row['scraped_text']

        if not article_text.strip():  # Verifica si el texto está vacío
            print(f"Artículo vacío para la URL: {row['link']}, omitiendo análisis.")
            df.at[index, 'SEO Analysis'] = "No se pudo scrapear el texto del artículo"
            continue

        # Llamada a GPT-4 para analizar cada artículo
        print(f"Procesando artículo {index + 1}/{len(df)}: {row['link']}")
        summary_and_seo_analysis = gpt4_summarize_and_analyze(article_text)
        df.at[index, 'SEO Analysis'] = summary_and_seo_analysis
        time.sleep(2)  # Agrega un pequeño retraso para no sobrecargar la API

    # Guardar el DataFrame a CSV
    output_csv = "GPT4_SEO_Analysis_on_SERP_Links_Text.csv"
    df.to_csv(output_csv, index=False)
    print(f"Archivo CSV generado: {output_csv}")

    return df

# Llama a la función para procesar los artículos scrapeados y generar el CSV
df = process_scraped_articles(df)

Procesando artículo 1/10: https://kdp.amazon.com/es_ES/
Procesando artículo 2/10: https://www.bookmundo.com/es/publica-tu-libro/autopublicar/
Procesando artículo 3/10: https://marketingonlineparaescritores.com/errores-autopublicar-libro/
Procesando artículo 4/10: https://www.kdpcommunity.com/s/question/0D58V00008iZvRqSAK/autopublicar-un-libro-ya-publicado-en-editorial?language=en_US
Artículo vacío para la URL: https://www.facebook.com/events/d41d8cd9/autopublicar-libros-en-puerto-rico/944034509789645/, omitiendo análisis.
Procesando artículo 6/10: https://www.escritores.org/publicar/1275-como-autopublicar-un-libro-de-forma-independiente-explicado-paso-a-paso
Procesando artículo 7/10: https://www.entrepreneur.com/es/emprendedores/deberias-autopublicar-tu-libro/422073
Procesando artículo 8/10: https://littleshopofstories.com/book/9798869012067
Procesando artículo 9/10: https://www.shopify.com/es/blog/autopublicar-un-libro
Artículo vacío para la URL: https://twitter.com/Fundeu/status/5666

In [12]:
def generar_guia_contenido(df):
    try:
        if 'SEO Analysis' not in df.columns:
            raise ValueError("La columna 'SEO Analysis' no está presente en el DataFrame.")
        
        analisis_seo = df['SEO Analysis'].tolist()
        texto_completo = ' '.join(analisis_seo)
        
        # Imprimir el texto completo que se enviará al modelo
        print("Texto completo enviado al modelo:")
        print(texto_completo)
        
        response = client.chat.completions.create(
            model="gpt-3.5-turbo-0125",
            messages=[
                {"role": "system", "content": "Eres un experto en SEO y creación de contenido. Tu tarea es crear una guía detallada para un escritor profesional que está preparando un artículo long-form."},
                {"role": "user", "content": f"Tomando como referencia el análisis SEO de los diez primeros resultados de Google SERP proporcionado en la columna 'SEO Analysis' del data frame, genera una guía detallada para un escritor profesional que está preparando un artículo long-form. La guía debe incluir:\n\n1. H1 y H2 sugeridos para el artículo\n2. Palabras clave, \n3. Estructura sugerida para el artículo\n4. Consejos de SEO específicos para este tema\n5. Sugerencias para meta tags y descripción\n6. Ideas para enlaces internos y externos\n\n. No hacer referencia a marcas, logos o productos específicos. Análisis SEO: {texto_completo}"}
            ]
        )
        
        guia_contenido = response.choices[0].message.content
        
        with open('guia_contenido_long_form.md', 'w', encoding='utf-8') as f:
            f.write(guia_contenido)
        
        print("Guía de contenido generada y guardada en 'guia_contenido_long_form.md'")
        return guia_contenido
    
    except Exception as e:
        print(f"Error al generar la guía de contenido: {str(e)}")
        return None

# Llamar a la función para generar la guía de contenido
guia_contenido = generar_guia_contenido(df)

Texto completo enviado al modelo:
Without the complete text of the article, a thorough summary and SEO examination can't be accurately performed. However, based on the provided segment, here are some general observations:

Keywords: 
The primary keywords in this extract include 'Gane', '50%', 'Kindle Vella', 'autores estadounidenses', 'inglés'. These keywords are relevant to the article's topic and would likely align the piece with searches about earning from Kindle Vella, especially for American authors writing in English. 

Biagrams:
Potential biagrams (two-word phrases) based on this extract could be 'Kindle Vella', '50% de', 'autores estadounidenses', and 'escriban en'. Including these biagrams in the text can help boost its SEO ranking when these particular phrases are searched for. 

Triagrams:
Potential triagrams (three-word phrases) could include 'Gane hasta un', 'en Kindle Vella', 'escriban en inglés'. These could help capture long-tail search queries related to earning from K